In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()
print(f"saving figures:{SAVE_FIGURES}")

In [ ]:
import numpy as np
import pandas as pd
from scipy import signal

NB_ID = "23"

In [ ]:
print("Loading Mixed Dataset...")
mixed_train_dir = get_unet_path(STAGE_MIXED, split=TRAIN)
print(f"Looking for data in: {mixed_train_dir}")

mixed_data = np.load(mixed_train_dir / f"{MIXED_DATASET}.npy")
df_meta = pd.read_csv(mixed_train_dir / f"{MIXED_METADATA}.csv")

print(f"Tensor Shape: {mixed_data.shape}")
print(f"Metadata Rows: {len(df_meta)}")
assert len(mixed_data) == len(df_meta), "CRITICAL: Data and Metadata length mismatch!"

In [ ]:
# SINR Distribution
plt.figure(figsize=(12, 5))
sns.histplot(df_meta['sinr_db'], bins=40, kde=True, color='teal', element="step")
plt.axvline(0, color='r', linestyle='--', label='0 dB (Boundary)')
plt.axvline(10, color='orange', linestyle='--', label='10 dB (Boundary)')
plt.title("SINR Distribution")
plt.xlabel("SINR (dB)")
plt.legend()
plt.show()

# Print exact percentages
print("SINR Distribution Stats:")
print(f"Severe (< 0 dB): {len(df_meta[df_meta['sinr_db'] < 0]) / len(df_meta):.1%}")
print(f"Medium (0-10 dB): {len(df_meta[(df_meta['sinr_db'] >= 0) & (df_meta['sinr_db'] < 10)]) / len(df_meta):.1%}")
print(f"Clean (> 10 dB): {len(df_meta[df_meta['sinr_db'] >= 10]) / len(df_meta):.1%}")

In [ ]:
# Interference Type Balance
plt.figure(figsize=(8, 4))
sns.countplot(data=df_meta, x='interference_type')
plt.title("Interference Source Balance")
plt.show()

In [ ]:
def plot_time_domain(idx):
    row = df_meta.iloc[idx]
    sig = mixed_data[idx]
    
    # Extract metadata for title
    sinr = row['sinr_db']
    itype = row['interference_type']
    source = row['source']
    
    plt.figure(figsize=(14, 4))
    # Plot I and Q
    plt.plot(sig[0, :1000], label='In-Phase (I)', alpha=0.9) # Zoom in on first 1000 samples
    plt.plot(sig[1, :1000], label='Quadrature (Q)', alpha=0.9)
    
    plt.title(f"Sample {idx} | Type: {itype} ({source}) | SINR: {sinr:.2f} dB")
    plt.xlabel("Time (Samples)")
    plt.ylabel("Amplitude")
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.show()

# Pick 3 specific examples to tell the story
print("--- VISUAL INSPECTION ---")
# The "destroyed" signal
idx_bad = df_meta[df_meta['sinr_db'] < -8].index[0]
plot_time_domain(idx_bad)

# The "struggle" signal
idx_mid = df_meta[(df_meta['sinr_db'] > -1) & (df_meta['sinr_db'] < 1)].index[0]
plot_time_domain(idx_mid)

# The "clean" signal
idx_good = df_meta[df_meta['sinr_db'] > 18].index[0]
plot_time_domain(idx_good)

In [ ]:
def plot_diagnostics(idx):
    row = df_meta.iloc[idx]
    sig = mixed_data[idx]
    
    # Create Complex Signal for analysis
    sig_complex = sig[0] + 1j * sig[1]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Power Spectral Density (PSD)
    f, Pxx_den = signal.welch(sig_complex, fs=25e6, nperseg=1024)
    # Center the frequency axis (0 Hz in middle)
    f_shifted = np.fft.fftshift(f)
    Pxx_shifted = np.fft.fftshift(Pxx_den)
    
    ax1.semilogy(f_shifted/1e6, Pxx_shifted)
    ax1.set_title(f"PSD: {row['interference_type']} @ {row['sinr_db']} dB")
    ax1.set_xlabel("Frequency (MHz)")
    ax1.set_ylabel("Power/Frequency (dB/Hz)")
    ax1.grid(True)
    
    # Constellation Plot (IQ Scatter)
    # Subsample for speed (plot every 10th point)
    ax2.scatter(sig[0, ::10], sig[1, ::10], alpha=0.1, s=1, c='purple')
    ax2.set_title("Constellation Diagram (Degraded)")
    ax2.set_xlabel("In-Phase")
    ax2.set_ylabel("Quadrature")
    ax2.set_xlim(-3, 3)
    ax2.set_ylim(-3, 3)
    ax2.grid(True)
    
    plt.show()

# Check a Spot Jammer (expect spikes)
spot_idx = df_meta[df_meta['interference_type'] == 'Spot'].index[0]
plot_diagnostics(spot_idx)

# Check a Digital Jammer (expect structured spectrum)
dig_idx = df_meta[df_meta['interference_type'] == 'Digital'].index[0]
plot_diagnostics(dig_idx)